In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.0 MB/s eta 0:00:00a 0:00:01


In [2]:
!rm -rf /kaggle/working/*

In [3]:
import torch
import gc
import os

print("Before cleanup:")
print("GPU:", torch.cuda.memory_allocated() / 1024**2, "MB")

# =========================
# DELETE VARIABLES
# =========================
for var in list(globals().keys()):
    if var not in ["torch", "gc", "os"]:
        del globals()[var]

# =========================
# PYTHON GARBAGE COLLECT
# =========================
gc.collect()

# =========================
# CLEAR CUDA CACHE
# =========================
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

# =========================
# OPTIONAL: RESET PEAK STATS
# =========================
torch.cuda.reset_peak_memory_stats()

print("After cleanup:")
print("GPU:", torch.cuda.memory_allocated() / 1024**2, "MB")

Before cleanup:
GPU: 0.0 MB
After cleanup:
GPU: 0.0 MB


In [4]:
import os
import json
import random
import shutil
from tqdm import tqdm
import torch
import zipfile

# =========================
# CONFIG
# =========================
ROOT = "/kaggle/input/datasets/qianlanzz/xbd-dataset/xbd/tier1"
OUTPUT = "/kaggle/working/final_seg_dataset"

TOTAL_IMAGES = 500      # 🔥 smaller = easier overfit
SEED = 42

random.seed(SEED)

# =========================
# WKT PARSER
# =========================
def parse_wkt_polygon(wkt):
    try:
        wkt = wkt.replace("MULTIPOLYGON (((", "").replace(")))", "")
        wkt = wkt.replace("POLYGON ((", "").replace("))", "")

        polygons = wkt.split(")), ((")
        coords = []

        for poly in polygons:
            for p in poly.split(","):
                x, y = map(float, p.strip().split())
                coords.append((x, y))

        return coords[:-1] if len(coords) > 1 else coords
    except:
        return []

# =========================
# DIR SETUP
# =========================
for split in ["train"]:
    os.makedirs(f"{OUTPUT}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUTPUT}/labels/{split}", exist_ok=True)

# =========================
# LOAD DATA
# =========================
label_dir = f"{ROOT}/labels"

ann_files = [
    os.path.join(label_dir, f)
    for f in os.listdir(label_dir)
    if f.endswith("_post_disaster.json")
]

print(f"Total Tier1 POST samples: {len(ann_files)}")

data = []

for path in tqdm(ann_files):
    with open(path) as f:
        js = json.load(f)

    img_name = js["metadata"]["img_name"]
    img_path = os.path.join(ROOT, "images", img_name)

    if not os.path.exists(img_path):
        continue

    objs = []
    for obj in js["features"]["xy"]:
        props = obj["properties"]

        if props.get("feature_type") != "building":
            continue

        dmg = props.get("subtype")
        if dmg not in ["no-damage", "minor-damage", "major-damage", "destroyed"]:
            continue

        coords = parse_wkt_polygon(obj["wkt"])
        if len(coords) < 3:
            continue

        objs.append(obj)

    if not objs:
        continue

    data.append({
        "image": img_path,
        "objects": objs,
        "h": js["metadata"]["height"],
        "w": js["metadata"]["width"]
    })

print(f"Usable samples: {len(data)}")

# =========================
# SAMPLING (SMALL = OVERFIT)
# =========================
selected = random.sample(data, min(TOTAL_IMAGES, len(data)))
random.shuffle(selected)

# =========================
# CLASS MAP
# =========================
CLASS_MAP = {
    "no-damage": 0,
    "minor-damage": 1,
    "major-damage": 2,
    "destroyed": 3
}

# =========================
# YOLO FORMAT (TRAIN ONLY)
# =========================
print("Creating dataset...")

valid_count = 0

for d in tqdm(selected):
    img_path = d["image"]

    h, w = d["h"], d["w"]
    lines = []

    for obj in d["objects"]:
        dmg = obj["properties"]["subtype"]
        cls = CLASS_MAP[dmg]

        coords = parse_wkt_polygon(obj["wkt"])
        if len(coords) < 3:
            continue

        norm = []
        for x, y in coords:
            norm.append(x / w)
            norm.append(y / h)

        if len(norm) < 6:
            continue

        lines.append(f"{cls} " + " ".join(map(str, norm)))

    if lines:
        out_img = f"{OUTPUT}/images/train/{os.path.basename(img_path)}"
        label_path = f"{OUTPUT}/labels/train/{os.path.basename(img_path).replace('.png','.txt')}"

        shutil.copy(img_path, out_img)

        with open(label_path, "w") as f:
            f.write("\n".join(lines))

        valid_count += 1

print(f"Final dataset size: {valid_count}")

# =========================
# YAML (TRAIN = VAL)
# =========================
with open(f"{OUTPUT}/data.yaml", "w") as f:
    f.write(f"""
path: {OUTPUT}
train: images/train
val: images/train

names:
  0: no-damage
  1: minor-damage
  2: major-damage
  3: destroyed
""")

# =========================
# TRAIN (OVERFIT MODE)
# =========================
from ultralytics import YOLO

print("GPUs available:", torch.cuda.device_count())
device = [0, 1] if torch.cuda.device_count() >= 2 else 0

model = YOLO("yolov8m-seg.pt")   # 🔥 bigger model

model.train(
    data=f"{OUTPUT}/data.yaml",
    epochs=150,                  # 🔥 high epochs
    imgsz=800,                  # 🔥 high resolution
    batch=10 if device != 0 else 6,
    augment=False,               # 🔥 OFF
    device=device,
    workers=4,
    cache=True,
    amp=True,
    close_mosaic=0,              # 🔥 OFF
    project=OUTPUT,
    name="seg_overfit"
)

# =========================
# SAVE ARTIFACTS
# =========================
print("Saving artifacts...")

BEST_MODEL = f"{OUTPUT}/seg_overfit/weights/best.pt"
ZIP_PATH = "/kaggle/working/seg_overfit_bundle.zip"

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:

    if os.path.exists(BEST_MODEL):
        z.write(BEST_MODEL, arcname="best.pt")

    z.write(f"{OUTPUT}/data.yaml", arcname="data.yaml")

    for root, _, files in os.walk(f"{OUTPUT}/images/train"):
        for f in files[:50]:
            path = os.path.join(root, f)
            z.write(path, arcname=f"sample_images/{f}")

print(f"ZIP saved at: {ZIP_PATH}")

Total Tier1 POST samples: 2799


100%|██████████| 2799/2799 [00:46<00:00, 59.57it/s]


Usable samples: 2241
Creating dataset...


100%|██████████| 500/500 [00:21<00:00, 23.80it/s]


Final dataset size: 500
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPUs available: 2
Ultralytics 8.4.34 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=10, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=0, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/final_seg_dataset/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=F